# STAC AI Workshop · NYC · 21 October 2026

## 05 · Did the signal predict the future—or read it?

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Compare the same opportunities under valid and leaked information
2. Recompute returns after explicit costs
3. Enforce availability before a simulated order
4. Distinguish a constructed profit from market alpha

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## Class Exercise · twelve independent opportunities
This matches **Web Section 5**. Each opportunity receives $10,000, independently; profits are summed, not compounded. BUY is held for five sessions. CASH means no trade and earns zero. Fees apply twice to initial notional.

The future-derived signal deliberately reads the realized return. This construction demonstrates leakage; it is not an investment strategy or a measured LLM output.

In [2]:
data=load_fixture("section5-replay.json")
results={"Available at the time":financial_replay(data,"valid",True),"Future-derived / gate OFF":financial_replay(data,"leaked",False),"Future-derived / gate ON":financial_replay(data,"leaked",True)}
show_table([{"Setup":k,"Trades":v["trades"],"Rejected signals":v["blocked"],"Net P&L ($)":round(v["net_pnl"],2),"Net bp/opportunity":round(v["net_bps"],2),"Worst opportunity ($)":round(v["worst_pnl"],2)} for k,v in results.items()])
assert results["Future-derived / gate ON"]["trades"]==0
assert results["Future-derived / gate ON"]["blocked"]==len(data["observations"])
print("No trade (cash) benchmark: $0. No interest, annualization or compounding.")

Setup,Trades,Rejected signals,Net P&L ($),Net bp/opportunity,Worst opportunity ($)
Available at the time,7,0,-50.0,-4.17,-390.0
Future-derived / gate OFF,7,0,870.0,72.5,0
Future-derived / gate ON,0,12,0,0.0,0


No trade (cash) benchmark: $0. No interest, annualization or compounding.


## Inspect the ledger
The observed return is used for scoring only. The invalid future-derived signal uses it to choose the action, creating look-ahead leakage. The gate uses availability metadata, not future returns.

In [3]:
replay=results["Future-derived / gate OFF"]
show_table([{"Opportunity":r["id"],"Decision":r["decision_time"][:10],"Signal available":r["available"][:10],"Signal":r["signal"],"Time eligible":r["eligible"],"Realized return (%)":round(100*r["realized_return"],2),"Net P&L ($)":round(r["pnl"],2)} for r in replay["rows"]])

Opportunity,Decision,Signal available,Signal,Time eligible,Realized return (%),Net P&L ($)
week-01,2026-01-05,2026-01-10,BUY,False,1.8,150.0
week-02,2026-01-12,2026-01-17,CASH,False,-2.4,0
week-03,2026-01-19,2026-01-24,BUY,False,1.1,80.0
week-04,2026-01-26,2026-01-31,CASH,False,-0.9,0
week-05,2026-02-02,2026-02-07,BUY,False,3.2,290.0
week-06,2026-02-09,2026-02-14,CASH,False,-1.7,0
week-07,2026-02-16,2026-02-21,BUY,False,0.7,40.0
week-08,2026-02-23,2026-02-28,CASH,False,-3.6,0
week-09,2026-03-02,2026-03-07,BUY,False,2.1,180.0
week-10,2026-03-09,2026-03-14,BUY,False,0.4,10.0


## Change trading costs
For a BUY, $r_{net}=r_{realized}-2f/10{,}000$, where f is the fee in basis points **each way**. This simple convention applies both fees to initial notional; it is not the earlier four-way experiment's exact exit-notional fee convention. Keep conventions explicit when comparing results.

In [4]:
show_table([{"Fee each way (bp)":fee,"Time-valid net P&L ($)":round(financial_replay(data,"valid",True,fee)["net_pnl"],2),"Leaked / unguarded P&L ($)":round(financial_replay(data,"leaked",False,fee)["net_pnl"],2)} for fee in (0,15,30)])
references=load_fixture("section567-reference.json")["financial"]
for ref in references:
    got=financial_replay(data,ref["mode"],ref["gate"],ref["fee"])
    for key in ("trades","blocked","net_pnl","net_bps","worst_pnl"):
        assert math.isclose(got[key],ref["result"][key],abs_tol=1e-10)
    for a,b in zip(got["rows"],ref["result"]["rows"]):
        for key in ("signal","available","eligible","blocked","trade"): assert a[key]==b[key]
        assert math.isclose(a["net"],b["net"],abs_tol=1e-12)
print("PASS: all 12 mode/gate/fee combinations match the browser.")

Fee each way (bp),Time-valid net P&L ($),Leaked / unguarded P&L ($)
0,160.0,1080.0
15,-50.0,870.0
30,-260.0,660.0


PASS: all 12 mode/gate/fee combinations match the browser.


## Learning check
Does blocking future information guarantee a good strategy? **No.** It protects the information boundary. Prediction, realistic costs and robustness need separate evidence.

Does the lowest individual P&L estimate VaR? **No.** It is a descriptive worst opportunity in this small fixture.

A point-in-time archive is required to establish real availability; this exercise only checks supplied timestamps.

## Main takeaway
Freeze the information boundary before interpreting returns. An attractive backtest can be explained by leakage rather than prediction; a gate can remove that invalid advantage without creating alpha.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.